In [0]:
# Databricks notebook source
# Required imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,expr, when, explode, current_date,regexp_replace,date_format, sum as _sum, count as _count, month, year, avg as _avg, sha2,to_date
from pyspark.sql.types import IntegerType, StringType, DoubleType, DateType, ArrayType, StructType, StructField
import pandas as pd
import os

In [0]:
spark = SparkSession.builder\
        .appName('InsuranceCraft')\
        .config('Spark.sql.adaptive.enabled',True)\
        .config("Spark.dynamicAllocation.enables",True)\
        .getOrCreate()

In [0]:
import json

# Get the clusterAllTags string from Spark config
cluster_tags_json_str = spark.conf.get("spark.databricks.clusterUsageTags.clusterAllTags")

# Parse the JSON string into a Python list of dictionaries
cluster_tags = json.loads(cluster_tags_json_str)

# Helper function to extract tag value by key
def get_tag_value(key, tags):
    return next((tag["value"] for tag in tags if tag["key"] == key), None)

# Extract individual values
file_path = get_tag_value("FilePath", cluster_tags)
host = get_tag_value("host", cluster_tags)
database = get_tag_value("database", cluster_tags)

# Output the result
print("File path:", file_path)
print("Host:", host)
print("Database:", database)

In [0]:
# List available secret scopes
dbutils.secrets.listScopes()

# Retrieve individual secrets from the scope "mssqlScope"
mssql_username = dbutils.secrets.get(scope="mssqlScope", key="Username-mssql")
mssql_password = dbutils.secrets.get(scope="mssqlScope", key="mssql-password")

# Print to verify (not recommended in production due to security)
print("Username:", mssql_username)
print("Password:", mssql_password)

In [0]:
#from pyspark.sql.types import StructType, StructField, StringType, ArrayType

schema = StructType([
    StructField("_id", StructType([
        StructField("$oid", StringType(), True)
    ]), True),
    StructField("customer_code", StructType([
        StructField("$numberInt", StringType(), True)
    ]), True),
    StructField("age", StructType([
        StructField("$numberInt", StringType(), True)
    ]), True),
    StructField("age_group", StringType(), True),
    StructField("city", StringType(), True),
    StructField("acquisition_channel", StringType(), True),
    StructField("policy_id", StringType(), True),
    StructField("base_coverage_amt", StructType([
        StructField("$numberInt", StringType(), True)
    ]), True),
    StructField("base_premium_amt", StructType([
        StructField("$numberInt", StringType(), True)
    ]), True),
    StructField("sale", StructType([
        StructField("sale_id", StructType([
            StructField("$numberInt", StringType(), True)
        ]), True),
        StructField("sale_date", StringType(), True),
        StructField("final_premium_amt", StructType([
            StructField("$numberInt", StringType(), True)
        ]), True),
        StructField("sales_mode", StringType(), True)
    ]), True),
    StructField("revenue", StructType([
        StructField("revenue_id", StructType([
            StructField("$numberInt", StringType(), True)
        ]), True),
        StructField("revenue_date", StringType(), True),
        StructField("revenue_amt", StructType([
            StructField("$numberInt", StringType(), True)
        ]), True)
    ]), True),
    StructField("claims", ArrayType(StructType([
        StructField("$numberInt", StringType(), True)
    ])), True),
    StructField("date", StringType(), True),
    StructField("day", StructType([
        StructField("$numberInt", StringType(), True)
    ]), True),
    StructField("month", StructType([
        StructField("$numberInt", StringType(), True)
    ]), True),
    StructField("year", StructType([
        StructField("$numberInt", StringType(), True)
    ]), True),
    StructField("day_type", StringType(), True),
    StructField("pii_fields", StructType([
        StructField("pan_number", StringType(), True),
        StructField("aadhaar_number", StringType(), True)
    ]), True),
    StructField("injection_date", StringType(), True)
])

# Read the JSON file using the defined schema
df = spark.read.format("json").schema(schema).load(f'{file_path}/inbound/shield.json')
df.display()

In [0]:
def encrypt_sensitive_fields(df, input_list_pii_field):
    for field in input_list_pii_field:
        encrypted_field_name = f"{field}_encrypted"
        df = df.withColumn(encrypted_field_name, sha2(col(field), 256).cast(StringType()))
   
    # Drop the original PII fields
    df = df.drop(*input_list_pii_field)
   
    return df

In [0]:
flattened_df = df.withColumn("customer_code", col("customer_code.$numberInt").cast("int"))\
    .withColumn("_id", col("_id.$oid").cast("string"))\
    .withColumn("age", col("age.$numberInt").cast("int"))\
    .withColumn("base_coverage_amt", col("base_coverage_amt.$numberInt").cast("int"))\
    .withColumn("base_premium_amt", col("base_premium_amt.$numberInt").cast("int"))\
    .withColumn("sale_id", col("sale.sale_id.$numberInt").cast(StringType())) \
    .withColumn("sale_date", col("sale.sale_date").cast(StringType())) \
    .withColumn("final_premium_amt", col("sale.final_premium_amt.$numberInt").cast(IntegerType()))\
    .withColumn("sale_mode", col("sale.sales_mode").cast(StringType()))\
    .withColumn("revenue_id", col("revenue.revenue_id.$numberInt").cast(IntegerType())) \
    .withColumn("revenue_date", col("revenue.revenue_date").cast(StringType())) \
    .withColumn("revenue_amt", col("revenue.revenue_amt.$numberInt").cast(IntegerType()))\
    .withColumn("claims", col("claims.$numberInt").cast(ArrayType(IntegerType())))\
    .withColumn("day", col("day.$numberInt").cast(IntegerType())) \
    .withColumn("month", col("month.$numberInt").cast(IntegerType())) \
    .withColumn("year", col("year.$numberInt").cast(IntegerType()))\
    .withColumn("pan_number", col("pii_fields.pan_number").cast(StringType())) \
    .withColumn("aadhaar_number", col("pii_fields.aadhaar_number").cast(StringType())) \
    .drop("sale", "revenue", "pii_fields")
                     

# Explode the claims array to calculate KPIs
exploded_df = flattened_df.withColumn("claim_amount", explode("claims")).drop("claims")
exploded_df.show()

### encrypt PII fields by UDF
input_list_pii_field = ["pan_number", "aadhaar_number"]
# Apply the function
exploded_df = encrypt_sensitive_fields(exploded_df, input_list_pii_field)
exploded_df.show(truncate=False)

# ### Write the Data into bronze layer
bronze_path = f'{file_path}/outbound/bronze'
exploded_df.coalesce(1).write.format('csv').option('header', 'true').mode("overwrite").save(bronze_path)
exploded_df.display()

In [0]:
# Filter the Columns --> By Select Statement [ list of columnss]
columns = ["sale_id", "sale_date", "final_premium_amt", "sale_mode", "revenue_id", "revenue_date", 
           "revenue_amt", "customer_code", "age", "age_group", "city", "acquisition_channel", 
           "policy_id", "base_coverage_amt", "base_premium_amt", "date", "day", "month", 
           "year", "day_type", "claim_amount", "pan_number_encrypted", "aadhaar_number_encrypted"]

silverdf = exploded_df.select(*columns)
# Add New Columns --> By withColumn [as per requirement]
silverdf = silverdf.withColumn("etl_date", date_format(current_date(), "dd-MM-yyyy"))

# Handling nulls --> by fillna() & df.dropna()
silverdf = silverdf.fillna(0).fillna('')

# Handling Duplicate --> by dropDuplicates()
silverdf= silverdf.dropDuplicates()

# Removing Special Character &  --> by regex
silverdf = silverdf.withColumn("age", regexp_replace (col("age"), "[^a-zA-Z0-9 ]", " "))

### Write the Data into silver layer
silver_path = f'{file_path}/outbound/silver'
silverdf.coalesce(1).write.format('csv').option('header', 'true').mode("overwrite").save(silver_path)
silverdf.display()

In [0]:
# gold layer transformation
# Calculate KPIs: Total claims per policy, total claim amount, and average claim amount
total_claims_per_policy = silverdf.groupBy("policy_id").count().withColumnRenamed("count", "total_claims")
display(total_claims_per_policy)

total_claim_amount_per_policy = silverdf.groupBy("policy_id").agg(_sum("claim_amount").alias("total_claim_amount"))
display(total_claim_amount_per_policy)

avg_claim_amount_per_policy = silverdf.groupBy("policy_id").agg(_avg("claim_amount").alias("avg_claim_amount"))
display(avg_claim_amount_per_policy)

# Join KPIs with original DataFrame
transformed_df = silverdf.join(total_claims_per_policy, on="policy_id", how="left") \
                   .join(total_claim_amount_per_policy, on="policy_id", how="left") \
                   .join(avg_claim_amount_per_policy, on="policy_id", how="left")

display(transformed_df)

# Total Customers
total_customers = silverdf.select("customer_code").distinct().count()
print(f"Total Customers: The insurance company has a customer base of {total_customers}")


# Customer Distribution by Age Group
age_group_distribution = silverdf.groupBy("age_group").agg(_count("customer_code").alias("customer_count"))
display(age_group_distribution)

# Customer Acquisition Channels
acquisition_channel_distribution = silverdf.groupBy("acquisition_channel").agg(_count("customer_code").alias("customer_count"))
display(acquisition_channel_distribution)

# Customer Distribution by City
city_distribution = silverdf.groupBy("city").agg(_count("customer_code").alias("customer_count"))

# Customer Retention Rate (assuming previous data)
# Calculate retention rate for March
monthly_revenue_customers = (
    silverdf.withColumn(
        "clean_sale_date",
        to_date(
            when(col("sale_date") == "", None).otherwise(col("sale_date")),
            "yyyy-MM-dd"  # Adjust this format if your dates use a different pattern
        )
    )
    .groupBy(
        year(col("clean_sale_date")).alias("year"),
        month(col("clean_sale_date")).alias("month")
    )
    .agg(
        _sum("revenue_amt").alias("total_revenue"),
        _count("customer_code").alias("total_customers")
    )
)
display(monthly_revenue_customers)

# previous_month_data = monthly_revenue_customers.filter((col("year") == 2023) & (col("month") == 2)).first()
# previous_month_customers = previous_month_data["total_customers"]

# Revenue Generated in Last year of Business month
march_data = monthly_revenue_customers.filter((col("year") == 2023) & (col("month")==3))
display(march_data)

In [0]:
# Write the Data into Gold layer - PART 1
gold_path_claim_policy = f'{file_path}/outbound/gold/claim_policy'
transformed_df.coalesce(1).write.format('csv').option('header', 'true').mode("overwrite").save(gold_path_claim_policy)
# Write the Data into Gold layer - PART 2
gold_path_age_group = f'{file_path}/outbound/gold/age_group'
age_group_distribution.coalesce(1).write.format('csv').option('header', 'true').mode("overwrite").save(gold_path_age_group)
# Write the Data into Gold layer - PART 3
gold_path_acquisition_channel = f'{file_path}/outbound/gold/acquisition_channel'
acquisition_channel_distribution.coalesce(1).write.format('csv').option('header', 'true').mode("overwrite").save(gold_path_acquisition_channel)
# Write the Data into Gold layer - PART 4
gold_path_march_data = f'{file_path}/outbound/gold/march_data'
march_data.coalesce(1).write.format('csv').option('header', 'true').mode("overwrite").save(gold_path_march_data)

In [0]:
port = "1433"
#Define MSSQL connection properties
mssql_properties = {
    "url": f"jdbc:sqlserver://{host}:{port};databaseName={database};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;",
    "user": mssql_username,
    "password": mssql_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Write the DataFrame to MSSQL
silverdf.write.format("jdbc").mode("overwrite").option("dbtable", "InsuranceTransactions").options(**mssql_properties).save()
transformed_df.write.format("jdbc").mode("overwrite").option("dbtable", "transformed_df").options(**mssql_properties).save()
age_group_distribution.write.format("jdbc").mode("overwrite").option("dbtable", "age_group_distribution").options(**mssql_properties).save()
acquisition_channel_distribution.write.format("jdbc").mode("overwrite").option("dbtable", "acquisition_channel_distribution").options(**mssql_properties).save()
march_data.write.format("jdbc").mode("overwrite").option("dbtable", "march_data").options(**mssql_properties).save()